# Notebook 04: RAG Pipeline

## Purpose
Wire all components from Notebooks 01–03 into a single end-to-end pipeline and validate that textbook-grounded RAG improves LLM accuracy on MedQA compared to ano-retrieval baseline.

## Pipeline
```
Raw query
  -> SpaCy NER (en_core_sci_md)      extract clinical entities
  -> Query rewriting                  entity string replaces raw vignette
  -> FAISS retrieval                  top-k textbook chunks + confidence bands
  -> Specialty classifier             predicted domain added to prompt
  -> Prompt construction              structured prompt with hedging
  -> HuggingFace LLM (4-bit nf4)     grounded answer
```

## Scope
Validates the pipeline using the default model (`qwen3-4b`). Full 5-model benchmarking is in notebook 06.

## Benchmark reference
AMG-RAG: 74.1% F1 on MedQA using GPT-4o-mini + dynamic
PubMed knowledge graph.

## Outputs
```
models/rag/results.parquet   per-question evaluation results
models/rag/config.json       run settings and aggregate metrics
```

## 0. Setup

In [ ]:
# ── COLAB ONLY ────────────────────────────────────────────────────────────────
# Prerequisites in Colab Secrets (key icon in sidebar):
#   GITHUB_REPO_URL  — repo clone URL
#   HF_TOKEN         — huggingface.co/settings/tokens
#
# Accept gated model terms once at these URLs before running:
#   MedGemma : https://huggingface.co/google/medgemma-4b-it
#   Gemma 3  : https://huggingface.co/google/gemma-3-4b-it
#   Ministral: https://huggingface.co/mistralai/Ministral-3B-Instruct-2410
# ─────────────────────────────────────────────────────────────────────────────

import os, shutil
from pathlib import Path
from google.colab import drive, userdata

HF_TOKEN = userdata.get('HF_TOKEN')

if not Path('/content/emma').exists():
    !git clone https://github.com/jaxendutta/emma.git
os.chdir('/content/emma')

!pip install -e . -q
!pip install transformers accelerate bitsandbytes faiss-cpu sentence-transformers -q
!pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_md-0.5.4.tar.gz -q

drive.mount('/content/drive')
DRIVE_BASE = Path('/content/drive/MyDrive/emma/models')
DRIVE_OUT  = DRIVE_BASE / 'rag'
DRIVE_OUT.mkdir(parents=True, exist_ok=True)

# Restore artifacts produced by notebooks 01-03
for artifact in ['vectorstore', 'classifier', 'clustering', 'rag']:
    dst = Path(f'/content/emma/models/{artifact}')
    dst.mkdir(parents=True, exist_ok=True)
    for f in (DRIVE_BASE / artifact).glob('*'):
        if f.is_file():
            shutil.copy(f, dst / f.name)
    print(f'  Restored {artifact}: {[f.name for f in dst.glob("*")]}')

import json, re, warnings
import matplotlib.pyplot as plt, numpy as np, pandas as pd
from tqdm import tqdm
from src.data import REPO_ROOT, load_medqa
from src.retrieval import EMMARetriever, extract_entities, get_default_model_id, rewrite_query
warnings.filterwarnings('ignore')
RAG_DIR = REPO_ROOT / 'models' / 'rag'
RAG_DIR.mkdir(parents=True, exist_ok=True)
print('✓ Colab setup complete.')

In [ ]:
import json
import os
import re
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm

from src.data import REPO_ROOT, load_medqa
from src.retrieval import EMMARetriever, extract_entities, get_default_model_id, rewrite_query

warnings.filterwarnings('ignore')

RAG_DIR = REPO_ROOT / 'models' / 'rag'
RAG_DIR.mkdir(parents=True, exist_ok=True)

# HF token -- required for gated models (MedGemma, Gemma 3, Ministral)
# Set via Colab Secrets or environment variable; never hardcode in the notebook
HF_TOKEN = os.environ.get('HF_TOKEN') or globals().get('HF_TOKEN') or None

# AMG-RAG benchmark reference
AMGRAG_F1  = 74.1   # MedQA F1,  GPT-4o-mini backbone + dynamic PubMed KG
AMGRAG_ACC = 66.34  # MedMCQA accuracy

print(f'Repo root : {REPO_ROOT}')
print(f'RAG dir   : {RAG_DIR}')
print(f'HF token  : {"set" if HF_TOKEN else "not set (gated models will fail)"}')

## 1. Load Pipeline

Loads the FAISS vectorstore, specialty classifier, and SpaCy NER from disk. The LLM is loaded lazily on the first call to `answer()` or `compare()`.

In [ ]:
retriever = EMMARetriever.load(hf_token=HF_TOKEN)

print(f'Model  : {retriever.model_id}')
print(f'Top-k  : {retriever.top_k}')
print(f'Index  : {retriever.index.ntotal:,} vectors')

## 2. NER Smoke Test

Verify that SpaCy extracts clinically useful entities and that the query rewriting mechanism correctly replaces vignette language with entity strings before embedding.

In [ ]:
test_queries = [
    'What is the mechanism of anaphylaxis?',
    'A 45-year-old man presents with chest pain, diaphoresis, and ST elevation in leads II, III, aVF.',
    'What are the side effects of metformin in type 2 diabetes?',
    'A child presents with inspiratory stridor, fever, and a barking cough. What is the treatment?',
]

rows = []
for q in test_queries:
    ents = extract_entities(q, retriever.nlp)
    rows.append({
        'Query':           q[:65] + '...' if len(q) > 65 else q,
        'Entities':        ents,
        'Rewrite applied': len(ents) > 0,
        'Rewritten query': ' '.join(ents) if ents else '(no rewrite)',
    })

display(pd.DataFrame(rows))

## 3. Retrieval Smoke Test

Test FAISS retrieval on a direct question and a clinical vignette to confirm that NER-based query rewriting improves retrieval on vignettes. Expected scores: 0.70+ for direct questions, 0.60-0.67 for vignettes (embedding dilution, which is a known behaviour).

In [ ]:
# Direct question
q = 'What is the mechanism of anaphylaxis?'
entities, rewritten, specialty, chunks = retriever.retrieve(q)

print(f'Query     : {q}')
print(f'Entities  : {entities}')
print(f'Rewritten : {rewritten}')
print(f'Specialty : {specialty}')
print()
for c in chunks:
    print(f'  [{c.confidence:8s}]  score={c.score:.4f}  {c.book}')
    print(f'             {c.text[:100].strip()}...')

In [ ]:
# Clinical vignette
vignette = (
    'A 58-year-old man with hypertension presents with crushing chest pain '
    'radiating to the left arm and diaphoresis. ECG shows ST elevation in V1-V4. '
    'What is the most appropriate immediate management?'
)
entities, rewritten, specialty, chunks = retriever.retrieve(vignette)

print(f'Entities  : {entities}')
print(f'Rewritten : {rewritten}')
print(f'Specialty : {specialty}')
print()
for c in chunks:
    print(f'  [{c.confidence:8s}]  score={c.score:.4f}  {c.book}')

## 4. End-to-End Demo

Run a single question through the full pipeline, RAG and baseline side by side, to confirm generation is working before batch evaluation.

In [ ]:
demo_q = 'What is the mechanism of action of beta blockers in heart failure?'

rag_result, base_result = retriever.compare(demo_q)

print('=' * 70)
print('WITH RAG')
print('=' * 70)
print(f'Entities  : {rag_result.entities}')
print(f'Specialty : {rag_result.specialty}')
print(f'Chunks    : {rag_result.metadata["n_chunks_retrieved"]}'
      f'  top score={rag_result.metadata["top_score"]:.4f}'
      f'  confidence={rag_result.metadata["top_confidence"]}')
print(f'Latency   : {rag_result.latency_s:.1f}s')
if rag_result.thinking:
    print(f'\nThinking (first 400 chars):\n{rag_result.thinking[:400]}...')
print(f'\nAnswer:\n{rag_result.answer}')

print('\n' + '=' * 70)
print('WITHOUT RAG (baseline)')
print('=' * 70)
print(f'Latency   : {base_result.latency_s:.1f}s')
print(f'\nAnswer:\n{base_result.answer}')

In [ ]:
# Full RAG prompt for inspection
print('FULL RAG PROMPT')
print('-' * 70)
print(rag_result.prompt)

## 5. Batch Evaluation

Evaluate RAG vs baseline on a sample of MedQA test questions. Each question is answered twice: once with retrieval, once without. Accuracy is the fraction of questions where the model selects the correct option (A/B/C/D).

<div class="alert alert-block alert-info">
<b style="font-size: 1.2em;">ⓘ Note</b>

This notebook validates the pipeline with the default model only. The full 5-model benchmark with 100 questions per model is in Notebook 06.

**AMG-RAG reference:** 74.1% F1 on MedQA (GPT-4o-mini + dynamic PubMed KG).
</div>

In [ ]:
def extract_answer_letter(response: str, options: dict) -> str | None:
    """
    Extract the predicted answer letter (A/B/C/D) from an LLM response.

    Tries patterns in priority order — most specific first, most general last.
    Strips residual thinking tokens before matching.
    Falls back to substring matching against option text if no letter found.
    """
    response = re.sub(r'<think>.*?</think>', '', response, flags=re.DOTALL).strip()

    patterns = [
        r'(?:answer is|answer:|correct answer is|correct option is)\s*:?\s*([A-D])',
        r'^\s*([A-D])[.):\s]',
        r'(?:therefore|thus|so|hence)[^A-D]{0,20}([A-D])\s+(?:is|would be|appears)',
        r'(?:option|choice)\s+([A-D])\b',
        r'\b([A-D])\s+is\s+(?:correct|the answer|most likely)',
        r'\b([A-D])\b',
    ]
    for pattern in patterns:
        m = re.search(pattern, response, re.IGNORECASE | re.MULTILINE)
        if m:
            return m.group(1).upper()

    for letter, text in options.items():
        if str(text).lower()[:40] in response.lower():
            return letter.upper()

    return None

In [ ]:
N_EVAL      = 50    # increase to 100+ for a more reliable estimate
RANDOM_SEED = 42

medqa_test = load_medqa(split='test')
eval_df    = medqa_test.sample(n=min(N_EVAL, len(medqa_test)), random_state=RANDOM_SEED).reset_index(drop=True)

print(f'Model          : {retriever.model_id}')
print(f'Questions      : {len(eval_df)}')
print(f'Total LLM calls: {len(eval_df) * 2}  (RAG + baseline per question)')
print(f'Est. runtime   : ~{len(eval_df) * 2.5 / 60:.0f} min at ~2.5 min/question')
print(f'AMG-RAG ref    : {AMGRAG_F1}% F1 on MedQA (GPT-4o-mini backbone)')
display(eval_df[['question', 'answer']].head(3))

In [ ]:
# ── Checkpoint-aware evaluation loop ─────────────────────────────────────────
# Saves progress to Drive after every question. If the runtime is interrupted,
# re-run this cell to resume from the last completed question automatically.
# ─────────────────────────────────────────────────────────────────────────────

import shutil

CHECKPOINT_FILE = RAG_DIR / 'checkpoint.parquet'
CHECKPOINT_DRIVE = DRIVE_OUT / 'checkpoint.parquet' if 'DRIVE_OUT' in globals() else None

def _save_checkpoint(rows: list, to_drive: bool = True) -> None:
    pd.DataFrame(rows).to_parquet(CHECKPOINT_FILE, index=False)
    if to_drive and CHECKPOINT_DRIVE is not None:
        try:
            shutil.copy(CHECKPOINT_FILE, CHECKPOINT_DRIVE)
        except Exception:
            pass  # Drive not mounted — skip silently

def _load_checkpoint() -> list:
    if CHECKPOINT_FILE.exists():
        df = pd.read_parquet(CHECKPOINT_FILE)
        print(f'> Resuming from checkpoint: {len(df)} questions already done.')
        return df.to_dict('records')
    return []

# Resume from checkpoint if one exists
rows = _load_checkpoint()
done_questions = {r['question'] for r in rows}

# Filter eval_df to only the remaining questions
remaining_df = eval_df[~eval_df['question'].str[:100].isin(done_questions)].copy()
print(f'> Questions remaining : {len(remaining_df)} / {len(eval_df)}')

if len(remaining_df) == 0:
    print('✓ All questions already evaluated.\n> Loading results from checkpoint...')
    results_df = pd.DataFrame(rows)
else:
    bar_fmt = '{l_bar}{bar}| {n:,}/{total:,} [{elapsed}<{remaining}]  {postfix}'
    pbar    = tqdm(remaining_df.iterrows(), total=len(remaining_df), unit=' q', bar_format=bar_fmt)

    for _, row in pbar:
        q        = row['question']
        options  = row['options']
        correct  = str(row['answer_idx']).upper()
        opts_str = '\n'.join(f"{k}. {v}" for k, v in options.items())
        full_q   = f"{q}\n\nOptions:\n{opts_str}"

        try:
            rag_res     = retriever.answer(full_q, use_rag=True)
            rag_pred    = extract_answer_letter(rag_res.answer, options)
            rag_correct = rag_pred == correct if rag_pred else False
        except Exception:
            rag_res, rag_pred, rag_correct = None, None, False

        try:
            base_res     = retriever.answer(full_q, use_rag=False)
            base_pred    = extract_answer_letter(base_res.answer, options)
            base_correct = base_pred == correct if base_pred else False
        except Exception:
            base_res, base_pred, base_correct = None, None, False

        rows.append({
            'question':       q[:100],
            'correct':        correct,
            'specialty':      rag_res.specialty if rag_res else None,
            'n_chunks':       rag_res.metadata.get('n_chunks_retrieved', 0) if rag_res else 0,
            'top_score':      rag_res.metadata.get('top_score') if rag_res else None,
            'top_confidence': rag_res.metadata.get('top_confidence') if rag_res else None,
            'rag_pred':       rag_pred,
            'rag_correct':    rag_correct,
            'rag_latency':    rag_res.latency_s if rag_res else None,
            'base_pred':      base_pred,
            'base_correct':   base_correct,
            'base_latency':   base_res.latency_s if base_res else None,
        })

        n        = len(rows)
        rag_acc  = sum(r['rag_correct']  for r in rows) / n * 100
        base_acc = sum(r['base_correct'] for r in rows) / n * 100
        pbar.set_postfix_str(f'RAG={rag_acc:.1f}%  Base={base_acc:.1f}%  Delta={rag_acc - base_acc:+.1f}%')

        # Save checkpoint to disk (and Drive if mounted) after every question
        _save_checkpoint(rows, to_drive=True)  # set to_drive=True in Colab

    results_df = pd.DataFrame(rows)
    print(f'\nDone. {len(results_df)} questions evaluated.')
    print(f'✓ Checkpoint saved -> {CHECKPOINT_FILE}')

## 6. Results

In [ ]:
rag_acc  = results_df['rag_correct'].mean()  * 100
base_acc = results_df['base_correct'].mean() * 100
delta    = rag_acc - base_acc
scores   = results_df['top_score'].dropna()

print('=' * 55)
print(f'  Model             : {retriever.model_id}')
print(f'  Questions         : {len(results_df)}')
print(f'  RAG accuracy      : {rag_acc:.1f}%')
print(f'  Baseline accuracy : {base_acc:.1f}%')
print(f'  Delta (RAG-base)  : {delta:+.1f}%')
print(f'  Mean top score    : {scores.mean():.4f}')
print('=' * 55)
print(f'  AMG-RAG F1 (MedQA)    : {AMGRAG_F1}%  [GPT-4o-mini + PubMed KG]')
print(f'  AMG-RAG acc (MedMCQA) : {AMGRAG_ACC}%')
print('  Note: gap reflects model size (3-4B vs ~8B) and retrieval strategy.')
print('=' * 55)

In [ ]:
# Accuracy by specialty
spec_df = (
    results_df.groupby('specialty')
    .agg(n=('rag_correct', 'count'),
         rag=('rag_correct', 'mean'),
         base=('base_correct', 'mean'))
    .assign(rag=lambda d:   (d.rag  * 100).round(1),
            base=lambda d:  (d.base * 100).round(1),
            delta=lambda d: (d.rag - d.base).round(1))
    .sort_values('n', ascending=False)
    .reset_index()
    .rename(columns={'rag': 'RAG acc%', 'base': 'Base acc%', 'delta': 'Delta'})
)
print('Accuracy by specialty:')
display(spec_df)

In [ ]:
# Retrieval quality
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(scores, bins=20, color='steelblue', edgecolor='white')
axes[0].axvline(scores.mean(), color='red', linestyle='--', label=f'Mean {scores.mean():.3f}')
axes[0].set(xlabel='Top retrieval score', ylabel='Count', title='Score distribution')
axes[0].legend()

conf_counts = results_df['top_confidence'].value_counts()
axes[1].bar(conf_counts.index, conf_counts.values, color='steelblue', edgecolor='white')
axes[1].set(xlabel='Confidence band', ylabel='Count', title='Top chunk confidence')

conf_acc = results_df.groupby('top_confidence')['rag_correct'].mean() * 100
axes[2].bar(conf_acc.index, conf_acc.values, color='steelblue', edgecolor='white')
axes[2].axhline(base_acc, color='red', linestyle='--', label=f'Baseline {base_acc:.1f}%')
axes[2].set(xlabel='Top chunk confidence', ylabel='RAG accuracy (%)', title='RAG acc by confidence')
axes[2].legend()

plt.suptitle(f'{retriever.model_id} — {len(results_df)} MedQA questions')
plt.tight_layout()
plt.show()

In [ ]:
# RAG helped vs hurt breakdown
rag_helped = results_df[ results_df['rag_correct'] & ~results_df['base_correct']]
rag_hurt   = results_df[~results_df['rag_correct'] &  results_df['base_correct']]
both_right = results_df[ results_df['rag_correct'] &  results_df['base_correct']]
both_wrong = results_df[~results_df['rag_correct'] & ~results_df['base_correct']]

print(f'RAG helped (correct w/ RAG, wrong w/o)  : {len(rag_helped)}')
print(f'RAG hurt   (wrong w/ RAG, correct w/o)  : {len(rag_hurt)}')
print(f'Both correct                             : {len(both_right)}')
print(f'Both wrong                               : {len(both_wrong)}')

if len(rag_helped) > 0:
    print('\nSample questions RAG helped with:')
    display(rag_helped[['question', 'correct', 'top_confidence', 'top_score']].head(3))

if len(rag_hurt) > 0:
    print('\nSample questions RAG hurt:')
    display(rag_hurt[['question', 'correct', 'top_confidence', 'top_score']].head(3))

## 7. Save

In [ ]:
results_path = RAG_DIR / 'results.parquet'
results_df.to_parquet(results_path, index=False)
print(f'Saved -> {results_path}  ({results_path.stat().st_size / 1024:.0f} KB)')

config = {
    'model':          retriever.model_id,
    'n_eval':         len(results_df),
    'top_k':          retriever.top_k,
    'rag_accuracy':   round(rag_acc, 2),
    'base_accuracy':  round(base_acc, 2),
    'delta':          round(delta, 2),
    'mean_top_score': round(float(scores.mean()), 4) if len(scores) > 0 else None,
    'rag_helped':     len(rag_helped),
    'rag_hurt':       len(rag_hurt),
    'amgrag_ref_f1':  AMGRAG_F1,
    'amgrag_ref_acc': AMGRAG_ACC,
    'amgrag_note':    'GPT-4o-mini backbone, dynamic PubMed KG — MedQA F1 / MedMCQA acc',
}
config_path = RAG_DIR / 'config.json'
config_path.write_text(json.dumps(config, indent=2))
print(f'✓ Saved -> {config_path}')

In [ ]:
# ── COLAB ONLY: copy outputs to Drive ─────────────────────────────────────────
import shutil
for f in RAG_DIR.glob('*'):
    if f.suffix in ('.parquet', '.json'):   # skip checkpoint if still partial
        shutil.copy(f, DRIVE_OUT / f.name)
        print(f'  {f.name}  ({f.stat().st_size/1024:.1f} KB) -> Drive')
# Clean up checkpoint once results are final
(DRIVE_OUT / 'checkpoint.parquet').unlink(missing_ok=True)
(RAG_DIR   / 'checkpoint.parquet').unlink(missing_ok=True)
print(f'✓ Done -> {DRIVE_OUT}')

## 8. Summary

In [ ]:
config = json.loads((RAG_DIR / 'config.json').read_text())

summary_df = pd.DataFrame([
    {'Item': 'Model',                    'Value': config['model']},
    {'Item': 'Questions evaluated',      'Value': str(config['n_eval'])},
    {'Item': 'RAG accuracy',             'Value': f"{config['rag_accuracy']:.1f}%"},
    {'Item': 'Baseline accuracy',        'Value': f"{config['base_accuracy']:.1f}%"},
    {'Item': 'Delta (RAG - baseline)',   'Value': f"{config['delta']:+.1f}%"},
    {'Item': 'Mean top retrieval score', 'Value': str(config['mean_top_score'])},
    {'Item': 'RAG helped',               'Value': str(config['rag_helped'])},
    {'Item': 'RAG hurt',                 'Value': str(config['rag_hurt'])},
    {'Item': 'AMG-RAG F1 (MedQA)',       'Value': f"{config['amgrag_ref_f1']:.1f}%"},
    {'Item': 'AMG-RAG acc (MedMCQA)',    'Value': f"{config['amgrag_ref_acc']:.2f}%"},
    {'Item': 'AMG-RAG note',             'Value': config['amgrag_note']},
])
display(summary_df)

<div class="alert alert-block alert-info">
Next > Notebook 05: Quiz Mode
</div>